##### array_sort()

- arranges the input array in **ascending order (by default)**.
- Works for **numbers, strings, timestamps, structs**.
- Use a `lambda` comparator for **descending or custom logic**.
- `Sorting` affects only `elements` inside the **array, not rows**.
- When you have **NaN** values in an array, the following applies.
  - For **double/float type**, NaN is considered greater than any **non-NaN elements**.
  - **Null** elements are positioned at the **end** of the **resulting array**.

In [0]:
from pyspark.sql.functions import col, array_sort

##### 1) Sorting an array of integers

In [0]:
data = [
    (1, [3, 1, 2, 0, 5, 6, 9]),
    (2, [5, 4, 6, 3, 2, 1]),
    (3, [10, 8, 9, 14]),
    (4, [7, 7, 6]),
    (5, [6, 2]),
    (6, [None]),
    (7, [None, None, None]),
    (8, [None, 1, 2]),
    (9, [])
]

df_int = spark.createDataFrame(data, ["id", "numbers"])
display(df_int)

id,numbers
1,"List(3, 1, 2, 0, 5, 6, 9)"
2,"List(5, 4, 6, 3, 2, 1)"
3,"List(10, 8, 9, 14)"
4,"List(7, 7, 6)"
5,"List(6, 2)"
6,List(null)
7,"List(null, null, null)"
8,"List(null, 1, 2)"
9,List()


In [0]:
df_int_sorted = df_int.withColumn("sorted_numbers", array_sort("numbers"))
display(df_int_sorted)

id,numbers,sorted_numbers
1,"List(3, 1, 2, 0, 5, 6, 9)","List(0, 1, 2, 3, 5, 6, 9)"
2,"List(5, 4, 6, 3, 2, 1)","List(1, 2, 3, 4, 5, 6)"
3,"List(10, 8, 9, 14)","List(8, 9, 10, 14)"
4,"List(7, 7, 6)","List(6, 7, 7)"
5,"List(6, 2)","List(2, 6)"
6,List(null),List(null)
7,"List(null, null, null)","List(null, null, null)"
8,"List(null, 1, 2)","List(1, 2, null)"
9,List(),List()


##### 2) Sorting an array of strings
- Strings are sorted `lexicographically`.

In [0]:
data = [
    (1, ["bajaj", "airforce", "check", "soldier", "google", "Google", "krishna"], ["Spark", "Java", "Azure Databricks", ""]),
    (2, ["dolphin", "catia", "edge", "chrome"], ["spark sql", "ADF"]),
    (3, ["anand", "basketball", "orchid"], ["ApacheSpark", "Python"]),
    (4, ["adarsh", "almond", "orphan"], ["PySpark", None, None]),
    (5, ["superman", None], ["SQL", "Databricks", "SQL", "python"]),
    (6, ["", None, "sandhya"], [None, None, None]),
    (7, ["diamond", "apple"], ["KSql", None, "SQL"]),
    (8, ["", ""], ["Git", "", ""])
]

from pyspark.sql.types import ArrayType, StringType, IntegerType, StructType, StructField

schema = StructType([
    StructField("id", StringType(), True),
    StructField("Technology", ArrayType(StringType()), True),
    StructField("NewTechnology", ArrayType(StringType()), True)
])

df_str = spark.createDataFrame(data, schema)
display(df_str)

id,Technology,NewTechnology
1,"List(bajaj, airforce, check, soldier, google, Google, krishna)","List(Spark, Java, Azure Databricks, )"
2,"List(dolphin, catia, edge, chrome)","List(spark sql, ADF)"
3,"List(anand, basketball, orchid)","List(ApacheSpark, Python)"
4,"List(adarsh, almond, orphan)","List(PySpark, null, null)"
5,"List(superman, null)","List(SQL, Databricks, SQL, python)"
6,"List(, null, sandhya)","List(null, null, null)"
7,"List(diamond, apple)","List(KSql, null, SQL)"
8,"List(, )","List(Git, , )"


In [0]:
df_str_sorted = df_str\
    .withColumn("sorted_Technology", array_sort("Technology")) \
    .withColumn("sorted_NewTechnology", array_sort("NewTechnology")) \
        .select("Technology", "sorted_Technology", "NewTechnology",  "sorted_NewTechnology")
display(df_str_sorted)

Technology,sorted_Technology,NewTechnology,sorted_NewTechnology
"List(bajaj, airforce, check, soldier, google, Google, krishna)","List(Google, airforce, bajaj, check, google, krishna, soldier)","List(Spark, Java, Azure Databricks, )","List(, Azure Databricks, Java, Spark)"
"List(dolphin, catia, edge, chrome)","List(catia, chrome, dolphin, edge)","List(spark sql, ADF)","List(ADF, spark sql)"
"List(anand, basketball, orchid)","List(anand, basketball, orchid)","List(ApacheSpark, Python)","List(ApacheSpark, Python)"
"List(adarsh, almond, orphan)","List(adarsh, almond, orphan)","List(PySpark, null, null)","List(PySpark, null, null)"
"List(superman, null)","List(superman, null)","List(SQL, Databricks, SQL, python)","List(Databricks, SQL, SQL, python)"
"List(, null, sandhya)","List(, sandhya, null)","List(null, null, null)","List(null, null, null)"
"List(diamond, apple)","List(apple, diamond)","List(KSql, null, SQL)","List(KSql, SQL, null)"
"List(, )","List(, )","List(Git, , )","List(, , Git)"


##### 3) Sorting in descending order

In [0]:
from pyspark.sql.functions import expr

df_desc = df_str\
    .withColumn("sorted_desc_Tech", expr("""array_sort(Technology, (left, right) ->
            CASE
                WHEN left < right THEN 1
                WHEN left > right THEN -1
                ELSE 0
            END)
            """)) \
    .withColumn("sorted_desc_NewTech", expr("""array_sort(NewTechnology, (left, right) ->
            CASE
                WHEN left < right THEN 1
                WHEN left > right THEN -1
                ELSE 0
            END
        )
    """)) \
        .select("Technology", "sorted_desc_Tech", "NewTechnology",  "sorted_desc_NewTech")
                

display(df_desc)

Technology,sorted_desc_Tech,NewTechnology,sorted_desc_NewTech
"List(bajaj, airforce, check, soldier, google, Google, krishna)","List(soldier, krishna, google, check, bajaj, airforce, Google)","List(Spark, Java, Azure Databricks, )","List(Spark, Java, Azure Databricks, )"
"List(dolphin, catia, edge, chrome)","List(edge, dolphin, chrome, catia)","List(spark sql, ADF)","List(spark sql, ADF)"
"List(anand, basketball, orchid)","List(orchid, basketball, anand)","List(ApacheSpark, Python)","List(Python, ApacheSpark)"
"List(adarsh, almond, orphan)","List(orphan, almond, adarsh)","List(PySpark, null, null)","List(PySpark, null, null)"
"List(superman, null)","List(superman, null)","List(SQL, Databricks, SQL, python)","List(python, SQL, SQL, Databricks)"
"List(, null, sandhya)","List(, null, sandhya)","List(null, null, null)","List(null, null, null)"
"List(diamond, apple)","List(diamond, apple)","List(KSql, null, SQL)","List(KSql, null, SQL)"
"List(, )","List(, )","List(Git, , )","List(Git, , )"


##### 4) Sorting array of structs

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType
from pyspark.sql.functions import struct

data = [
    (1, [{"name": "A", "score": 90}, {"name": "B", "score": 85}]),
    (2, [{"name": "C", "score": 70}, {"name": "D", "score": 95}]),
    (3, [{"name": "E", "score": 90}, {"name": "F", "score": 85}]),
    (4, [{"name": "G", "score": 70}, {"name": "H", "score": 95}]),
    (5, [{"name": "I", "score": 90}, {"name": "J", "score": 85}]),
    (6, [{"name": "K", "score": 70}, {"name": "L", "score": 95}]),
    (7, [{"name": "M", "score": 70}, {"name": "N", "score": 95}]),
    (8, [{"name": "O", "score": 90}, {"name": "P", "score": 85}]),
    (9, [{"name": "Q", "score": 70}, {"name": "R", "score": 95}]),
    (10, [{"name": "S", "score": 90}, {"name": "T", "score": 85}])
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("students", ArrayType(StructType([
        StructField("name", StringType(), True),
        StructField("score", IntegerType(), True)
    ])), True)
])

df_arr = spark.createDataFrame(data, schema)
display(df_arr)

id,students
1,"List(List(A, 90), List(B, 85))"
2,"List(List(C, 70), List(D, 95))"
3,"List(List(E, 90), List(F, 85))"
4,"List(List(G, 70), List(H, 95))"
5,"List(List(I, 90), List(J, 85))"
6,"List(List(K, 70), List(L, 95))"
7,"List(List(M, 70), List(N, 95))"
8,"List(List(O, 90), List(P, 85))"
9,"List(List(Q, 70), List(R, 95))"
10,"List(List(S, 90), List(T, 85))"


In [0]:
df_sorted = df_arr.withColumn(
    "sorted_students",
    array_sort("students")
)

display(df_sorted)

id,students,sorted_students
1,"List(List(A, 90), List(B, 85))","List(List(A, 90), List(B, 85))"
2,"List(List(C, 70), List(D, 95))","List(List(C, 70), List(D, 95))"
3,"List(List(E, 90), List(F, 85))","List(List(E, 90), List(F, 85))"
4,"List(List(G, 70), List(H, 95))","List(List(G, 70), List(H, 95))"
5,"List(List(I, 90), List(J, 85))","List(List(I, 90), List(J, 85))"
6,"List(List(K, 70), List(L, 95))","List(List(K, 70), List(L, 95))"
7,"List(List(M, 70), List(N, 95))","List(List(M, 70), List(N, 95))"
8,"List(List(O, 90), List(P, 85))","List(List(O, 90), List(P, 85))"
9,"List(List(Q, 70), List(R, 95))","List(List(Q, 70), List(R, 95))"
10,"List(List(S, 90), List(T, 85))","List(List(S, 90), List(T, 85))"


In [0]:
df_custom = df_arr.withColumn(
    "sorted_by_score",
    expr("""
        array_sort(students, (l, r) ->
            CASE
                WHEN l.score < r.score THEN -1
                WHEN l.score > r.score THEN 1
                ELSE 0
            END
        )
    """)
)

display(df_custom)

id,students,sorted_by_score
1,"List(List(A, 90), List(B, 85))","List(List(B, 85), List(A, 90))"
2,"List(List(C, 70), List(D, 95))","List(List(C, 70), List(D, 95))"
3,"List(List(E, 90), List(F, 85))","List(List(F, 85), List(E, 90))"
4,"List(List(G, 70), List(H, 95))","List(List(G, 70), List(H, 95))"
5,"List(List(I, 90), List(J, 85))","List(List(J, 85), List(I, 90))"
6,"List(List(K, 70), List(L, 95))","List(List(K, 70), List(L, 95))"
7,"List(List(M, 70), List(N, 95))","List(List(M, 70), List(N, 95))"
8,"List(List(O, 90), List(P, 85))","List(List(P, 85), List(O, 90))"
9,"List(List(Q, 70), List(R, 95))","List(List(Q, 70), List(R, 95))"
10,"List(List(S, 90), List(T, 85))","List(List(T, 85), List(S, 90))"


##### 5) sort timestamps in an array

In [0]:
from pyspark.sql.functions import to_timestamp

data = [
    (1, ["2024-01-03", "2024-01-01", "2024-01-02"]),
    (2, ["2024-01-05", "2024-01-04", "2024-01-03"]),
    (3, ["2024-01-07", "2024-01-06", "2024-01-05"]),
    (4, ["2024-01-09", "2024-01-08", "2024-01-07"]),
    (5, ["2024-01-11", "2024-01-10", "2024-01-09"]),
    (6, ["2024-01-13", "2024-01-12", "2024-01-11"]),
    (7, ["2024-01-15", "2024-01-14", "2024-01-13"])
]

df_ts = spark.createDataFrame(data, ["id", "dates"])
display(df_ts)
df_ts.printSchema()

id,dates
1,"List(2024-01-03, 2024-01-01, 2024-01-02)"
2,"List(2024-01-05, 2024-01-04, 2024-01-03)"
3,"List(2024-01-07, 2024-01-06, 2024-01-05)"
4,"List(2024-01-09, 2024-01-08, 2024-01-07)"
5,"List(2024-01-11, 2024-01-10, 2024-01-09)"
6,"List(2024-01-13, 2024-01-12, 2024-01-11)"
7,"List(2024-01-15, 2024-01-14, 2024-01-13)"


root
 |-- id: long (nullable = true)
 |-- dates: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
df_ts_ts = df_ts.withColumn("dates_ts", expr("transform(dates, x -> to_timestamp(x))")) \
                .withColumn("sorted_dates", array_sort("dates_ts"))

display(df_ts_ts)

id,dates,dates_ts,sorted_dates
1,"List(2024-01-03, 2024-01-01, 2024-01-02)","List(2024-01-03T00:00:00.000Z, 2024-01-01T00:00:00.000Z, 2024-01-02T00:00:00.000Z)","List(2024-01-01T00:00:00.000Z, 2024-01-02T00:00:00.000Z, 2024-01-03T00:00:00.000Z)"
2,"List(2024-01-05, 2024-01-04, 2024-01-03)","List(2024-01-05T00:00:00.000Z, 2024-01-04T00:00:00.000Z, 2024-01-03T00:00:00.000Z)","List(2024-01-03T00:00:00.000Z, 2024-01-04T00:00:00.000Z, 2024-01-05T00:00:00.000Z)"
3,"List(2024-01-07, 2024-01-06, 2024-01-05)","List(2024-01-07T00:00:00.000Z, 2024-01-06T00:00:00.000Z, 2024-01-05T00:00:00.000Z)","List(2024-01-05T00:00:00.000Z, 2024-01-06T00:00:00.000Z, 2024-01-07T00:00:00.000Z)"
4,"List(2024-01-09, 2024-01-08, 2024-01-07)","List(2024-01-09T00:00:00.000Z, 2024-01-08T00:00:00.000Z, 2024-01-07T00:00:00.000Z)","List(2024-01-07T00:00:00.000Z, 2024-01-08T00:00:00.000Z, 2024-01-09T00:00:00.000Z)"
5,"List(2024-01-11, 2024-01-10, 2024-01-09)","List(2024-01-11T00:00:00.000Z, 2024-01-10T00:00:00.000Z, 2024-01-09T00:00:00.000Z)","List(2024-01-09T00:00:00.000Z, 2024-01-10T00:00:00.000Z, 2024-01-11T00:00:00.000Z)"
6,"List(2024-01-13, 2024-01-12, 2024-01-11)","List(2024-01-13T00:00:00.000Z, 2024-01-12T00:00:00.000Z, 2024-01-11T00:00:00.000Z)","List(2024-01-11T00:00:00.000Z, 2024-01-12T00:00:00.000Z, 2024-01-13T00:00:00.000Z)"
7,"List(2024-01-15, 2024-01-14, 2024-01-13)","List(2024-01-15T00:00:00.000Z, 2024-01-14T00:00:00.000Z, 2024-01-13T00:00:00.000Z)","List(2024-01-13T00:00:00.000Z, 2024-01-14T00:00:00.000Z, 2024-01-15T00:00:00.000Z)"
